## 0. Environment check

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

volume_root = Path(os.environ.get("NIGHTS_WATCH_VOLUME_ROOT", "/mnt/nightswatch-poc"))
print(f"Modal volume root: {volume_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
gpu_count = torch.cuda.device_count()
print(f"torch.cuda.device_count(): {gpu_count}")
assert torch.cuda.is_available(), "GPU is required. Enable a GPU in Modal notebook settings."
for idx in range(gpu_count):
    props = torch.cuda.get_device_properties(idx)
    print(f"GPU {idx}: {props.name}, VRAM={props.total_memory / 1e9:.2f} GB")
if gpu_count < 2:
    print("[WARN] Only one GPU detected. Training will run, but Ultralytics multi-GPU DDP will not be available.")
else:
    print("[DONE] Multi-GPU runtime detected")


## 1. Install dependencies

In [ ]:
!pip install "ultralytics==8.3.*" onnxruntime-gpu==1.24.4 onnx==1.21.0 roboflow==1.3.3 gdown==6.0.0 pycocotools==2.0.11 kaggle==2.0.1 pandas==3.0.2 --quiet

import importlib.metadata as metadata
import onnxruntime as ort
import torch

packages = ["ultralytics", "onnxruntime-gpu", "onnx", "roboflow", "gdown", "pycocotools", "kaggle", "pandas"]
for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not found")
print(f"ONNX Runtime providers: {ort.get_available_providers()}")
print(f"Torch CUDA available: {torch.cuda.is_available()}")
print("[DONE] Dependencies installed")


## 2. Clone GitHub repository

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

volume_root = Path(os.environ.get("NIGHTS_WATCH_VOLUME_ROOT", "/mnt/nightswatch-poc"))
repo_url = os.environ.get(
    "NIGHTS_WATCH_REPO_URL", "https://github.com/AdityaChaudhary2913/NightsWatch-PoC.git"
)
repo_dir = volume_root / "NightsWatch"
output_dir = volume_root / "poc_outputs"
data_dir = volume_root / "datasets"
yolo_config_dir = volume_root / ".ultralytics"
volume_root.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
yolo_config_dir.mkdir(parents=True, exist_ok=True)
os.environ["NIGHTS_WATCH_VOLUME_ROOT"] = str(volume_root)
os.environ["YOLO_CONFIG_DIR"] = str(yolo_config_dir)
if repo_dir.exists():
    print(f"Repository already exists at {repo_dir}; refreshing from remote")
    status = subprocess.run(["git", "-C", str(repo_dir), "status", "--short"], text=True, capture_output=True, check=True)
    if status.stdout.strip():
        raise RuntimeError(
            "Mounted repo has local changes; please commit/stash them or remove the repo directory before rerunning."
        )
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", "HEAD"], check=True)
else:
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
%cd /mnt/nightswatch-poc/NightsWatch
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
commit = subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "--short", "HEAD"], text=True, capture_output=True, check=True).stdout.strip()
print(f"[DONE] Repository ready: {repo_dir}")
print(f"[DONE] Repo commit: {commit}")
print(f"[DONE] Dataset root: {data_dir}")
print(f"[DONE] Output root: {output_dir}")


## 3. Load prepared dataset manifest

In [ ]:
import json
from pathlib import Path

import pandas as pd

from utils.env import get_log_dir

log_dir = get_log_dir()
prep_manifest_path = log_dir / "prep_manifest.json"
if not prep_manifest_path.exists():
    raise FileNotFoundError(
        f"Prep manifest not found at {prep_manifest_path}. Run run_prep.ipynb first on the same mounted volume."
    )
prep_manifest = json.loads(prep_manifest_path.read_text(encoding="utf-8"))
required_paths = [
    prep_manifest["thermal_yaml"],
    prep_manifest["eo_yaml"],
    prep_manifest["ir_yaml"],
    prep_manifest["pair_manifest"],
]
missing = [path for path in required_paths if not Path(path).exists()]
if missing:
    raise FileNotFoundError(f"Prepared dataset files are missing: {missing}")
thermal_yaml = prep_manifest["thermal_yaml"]
eo_yaml = prep_manifest["eo_yaml"]
ir_yaml = prep_manifest["ir_yaml"]
pair_manifest = prep_manifest["pair_manifest"]
dataset_records = prep_manifest["dataset_records"]
display(pd.DataFrame(dataset_records))
print(f"[DONE] Loaded prep manifest: {prep_manifest_path}")
print(f"[DONE] Thermal dataset yaml: {thermal_yaml}")
print(f"[DONE] EO dataset yaml: {eo_yaml}")
print(f"[DONE] IR dataset yaml: {ir_yaml}")


## 4. Training - FLIR thermal (LWIR unimodal)

In [ ]:
import json
from dataclasses import replace
from pathlib import Path

from train.train_single import run_training_job
from train.training_configs import FLIR_THERMAL_CONFIG
from utils.env import get_log_dir

training_summary_path = get_log_dir() / "training.json"
if training_summary_path.exists():
    training_summaries = json.loads(training_summary_path.read_text(encoding="utf-8"))
else:
    training_summaries = {}

if "thermal" in training_summaries and Path(training_summaries["thermal"]["model_artifact"]).exists():
    thermal_train = training_summaries["thermal"]
    thermal_weights = thermal_train["model_artifact"]
    print(f"[DONE] Reusing existing thermal weights: {thermal_weights}")
else:
    thermal_config = replace(FLIR_THERMAL_CONFIG, name="flir_thermal_yolo11s")
    print(json.dumps(thermal_config.to_dict(), indent=2))
    thermal_train = run_training_job(thermal_yaml, thermal_config, artifact_name="thermal_best.pt")
    thermal_weights = thermal_train["model_artifact"]
    training_summaries["thermal"] = thermal_train
    training_summary_path.write_text(json.dumps(training_summaries, indent=2), encoding="utf-8")
print(f"[DONE] Thermal training ready: {thermal_weights}")


## 5. Training - EO unimodal (DroneVehicle RGB or LLVIP visible)

In [ ]:
from dataclasses import replace
from pathlib import Path

from train.training_configs import EO_UNIMODAL_CONFIG

if "eo" in training_summaries and Path(training_summaries["eo"]["model_artifact"]).exists():
    eo_train = training_summaries["eo"]
    eo_weights = eo_train["model_artifact"]
    print(f"[DONE] Reusing existing EO weights: {eo_weights}")
else:
    eo_config = replace(EO_UNIMODAL_CONFIG, name="eo_unimodal_yolo11s")
    print(json.dumps(eo_config.to_dict(), indent=2))
    eo_train = run_training_job(eo_yaml, eo_config, artifact_name="eo_best.pt")
    eo_weights = eo_train["model_artifact"]
    training_summaries["eo"] = eo_train
    training_summary_path.write_text(json.dumps(training_summaries, indent=2), encoding="utf-8")
print(f"[DONE] EO training ready: {eo_weights}")


## 6. Training - IR unimodal (DroneVehicle IR or LLVIP infrared)

In [ ]:
from dataclasses import replace
from pathlib import Path

from train.training_configs import IR_UNIMODAL_CONFIG

if "ir" in training_summaries and Path(training_summaries["ir"]["model_artifact"]).exists():
    ir_train = training_summaries["ir"]
    ir_weights = ir_train["model_artifact"]
    print(f"[DONE] Reusing existing IR weights: {ir_weights}")
else:
    ir_config = replace(IR_UNIMODAL_CONFIG, name="ir_unimodal_yolo11s")
    print(json.dumps(ir_config.to_dict(), indent=2))
    ir_train = run_training_job(ir_yaml, ir_config, artifact_name="ir_best.pt")
    ir_weights = ir_train["model_artifact"]
    training_summaries["ir"] = ir_train
    training_summary_path.write_text(json.dumps(training_summaries, indent=2), encoding="utf-8")
print(f"[DONE] IR training ready: {ir_weights}")


## 7. Benchmark - latency and mAP profiling

In [ ]:
import pandas as pd

from eval.benchmark import benchmark_model

benchmark_jobs = [
    ("thermal_yolo11s", thermal_weights, thermal_yaml),
    ("eo_yolo11s", eo_weights, eo_yaml),
    ("ir_yolo11s", ir_weights, ir_yaml),
]
benchmarks = []
for model_name, weights, dataset_yaml in benchmark_jobs:
    result = benchmark_model(weights, dataset_yaml, model_name=model_name)
    benchmarks.append(result)
    print(f"[DONE] Benchmark complete: {model_name}, FP32={result['fp32_ms']:.1f}ms, FP16={result['fp16_ms']:.1f}ms, ONNX={result['onnx_ms']:.1f}ms")

display(pd.DataFrame([
    {
        "model": item["model_name"],
        "mAP@0.5": item["map50"],
        "mAP@0.5:0.95": item["map50_95"],
        "FP32 ms": item["fp32_ms"],
        "FP16 ms": item["fp16_ms"],
        "ONNX ms": item["onnx_ms"],
        "Jetson INT8 est ms": item["projected_jetson_int8_ms"],
        "Mem GB": item["peak_memory_gb"],
    }
    for item in benchmarks
]))


## 8. Late-fusion baseline

In [ ]:
import pandas as pd

from eval.fusion_baseline import run_late_fusion

fusion_result = run_late_fusion(eo_weights, ir_weights, pair_manifest)
display(pd.DataFrame(fusion_result["table"]))
print("[DONE] Late-fusion baseline complete")


## 9. Detection visualisations

In [ ]:
from IPython.display import Image as IPyImage, display

from visualise.save_detections import save_detection_images

visualisation_jobs = [
    ("thermal_yolo11s", thermal_weights, thermal_yaml),
    ("eo_yolo11s", eo_weights, eo_yaml),
    ("ir_yolo11s", ir_weights, ir_yaml),
]
visualisation_paths = []
for model_name, weights, dataset_yaml in visualisation_jobs:
    visualisation_paths.extend(save_detection_images(weights, dataset_yaml, model_name, num_images=10))

for path in visualisation_paths:
    display(IPyImage(filename=path))
print(f"[DONE] Detection visualisations saved and displayed: {len(visualisation_paths)} images")


## 10. Generate report

In [ ]:
from pathlib import Path

from report.generate_report import generate_report
from utils.env import get_output_dir

report_path = generate_report()
print(report_path.read_text(encoding="utf-8"))

expected_paths = [
    report_path,
    get_output_dir() / "logs" / "benchmark.json",
    get_output_dir() / "logs" / "fusion.json",
    get_output_dir() / "logs" / "dataset_summary.json",
    Path(thermal_weights),
    Path(eo_weights),
    Path(ir_weights),
]
for path in expected_paths:
    print(("[DONE]" if Path(path).exists() else "[MISSING]"), Path(path))
print("[DONE] Report generated and output files confirmed")


## 11. Final file listing

In [ ]:
from pathlib import Path

from utils.env import get_output_dir

output_root = get_output_dir()
for path in sorted(output_root.rglob("*")):
    print(path.relative_to(output_root))
print("[DONE] Final poc_outputs listing complete")
